# Stage 0 — TF-IDF Semantic Scoring
**[STUDENT VERSION — fill in the blanks]**

- Input: `stage0_descriptions.csv` — 38 Vietnamese destination descriptions.
- Input: `stage0_anchors.csv` — 8 semantic feature anchors.
- Output: normalized semantic score matrix (38 × 8).

> 💡 Cells marked `# TODO` require you to fill in the code.


In [ ]:
import csv
import re
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
BASE_DIR = Path('.')
DESCRIPTIONS_CSV = BASE_DIR / 'stage0_descriptions.csv'
ANCHORS_CSV = BASE_DIR / 'stage0_anchors.csv'

def read_csv_rows(path):
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

description_rows = read_csv_rows(DESCRIPTIONS_CSV)
anchor_rows      = read_csv_rows(ANCHORS_CSV)

places       = [row['place']       for row in description_rows]
provinces    = [row['province']    for row in description_rows]
descriptions = [row['description'] for row in description_rows]
features     = [row['feature']     for row in anchor_rows]
anchors      = [row['anchor_text'] for row in anchor_rows]


In [ ]:
WORD_RE = re.compile(r"\b[\wÀ-ỹà-ỹĐđ]+\b", re.UNICODE)
word_counts = [len(WORD_RE.findall(text)) for text in descriptions]
bad_counts  = [(p, c) for p, c in zip(places, word_counts) if c < 150 or c > 250]
if bad_counts:
    raise ValueError(f'Descriptions outside 150-250 word range: {bad_counts}')

print(f'Destination count: {len(descriptions)}')
print(f'Word count range: {min(word_counts)}-{max(word_counts)}')


Destination count: 38
Word count range: 192-220


## 🔧 TODO 1 — TF-IDF Vectorizer

Tạo `TfidfVectorizer` với các tham số sau rồi fit_transform toàn bộ corpus.

**Corpus** = descriptions + anchors (ghép lại, fit chung 1 vectorizer).  
**Lý do:** Để descriptions và anchors dùng chung vocabulary space — dot product mới có nghĩa.

Tham số cần dùng:
- `lowercase=True`
- `sublinear_tf=True` — dùng log(1+tf) thay vì tf thuần
- `ngram_range=(1, 2)` — unigram + bigram
- `token_pattern=r"(?u)\b\w+\b"`


In [ ]:
# TODO 1: Tạo vectorizer, fit_transform corpus, tính raw cosine similarity

# Bước 1: Tạo corpus = descriptions + anchors
corpus = None  # ← thay None bằng code của bạn

# Bước 2: Khởi tạo TfidfVectorizer với đúng tham số
vectorizer = None  # ← thay None bằng TfidfVectorizer(...)

# Bước 3: fit_transform corpus → ma trận X
X = None  # ← thay None bằng vectorizer.fit_transform(corpus)

# Bước 4: Tính raw cosine similarity
# X[:len(descriptions)] = description vectors
# X[len(descriptions):] = anchor vectors
raw = None  # ← thay None bằng cosine_similarity(...)

print(f'Vocabulary size: {len(vectorizer.get_feature_names_out())}')
print(f'Raw matrix shape: {raw.shape}')  # phải là (38, 8)


## 🔧 TODO 2 — Min-Max Normalization per Feature

Raw cosine scores rất nhỏ (0.001–0.3) và khác nhau giữa các features.  
Cần normalize **từng cột riêng** về [0, 1].

**Tại sao per-feature?** Vì mỗi feature có baseline khác nhau.  
Normalize theo toàn bộ matrix sẽ làm features dominant át features yếu.

Công thức:
```
normalized[i, j] = (raw[i, j] - min_j) / (max_j - min_j)
```


In [ ]:
# TODO 2: Per-feature min-max normalization

mins   = None  # ← min của mỗi cột (axis=0)
maxs   = None  # ← max của mỗi cột (axis=0)
scores = np.zeros_like(raw)

for j in range(raw.shape[1]):
    denom = None  # ← maxs[j] - mins[j]
    # TODO: điền công thức normalize vào đây
    scores[:, j] = None  # ← (raw[:, j] - mins[j]) / denom if denom > 0 else 0.0

print(f'Score matrix shape: {scores.shape}')
print(f'Score range: {scores.min():.3f} – {scores.max():.3f}')  # phải là 0.0 – 1.0


In [ ]:
def write_csv(path, rows, headers):
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)

score_rows = []
raw_rows   = []
for i, place in enumerate(places):
    base = {'place': place, 'province': provinces[i]}
    score_rows.append({**base, **{feature: f'{scores[i, j]:.3f}'  for j, feature in enumerate(features)}})
    raw_rows.append(  {**base, **{feature: f'{raw[i, j]:.6f}'    for j, feature in enumerate(features)}})

headers = ['place', 'province'] + features
write_csv(BASE_DIR / 'stage0_tfidf_scores_rerun.csv', score_rows, headers)
write_csv(BASE_DIR / 'stage0_raw_cosine_rerun.csv',   raw_rows,   headers)
print('Saved.')


## 🔧 TODO 3 — Sanity Check

In ra top 5 điểm đến cho mỗi feature.  
Kết quả phải hợp lý — ví dụ: feature "beach" phải có Biển Nhật Lệ, Lăng Cô trong top 5.


In [ ]:
# TODO 3: In top 5 điểm đến cho mỗi feature
for j, feature in enumerate(features):
    # Gợi ý: dùng np.argsort(scores[:, j])[::-1][:5] để lấy index top 5
    top_idx = None  # ← thay None bằng code của bạn
    print(f"{feature}: " + ", ".join(places[i] for i in top_idx))
